# Buscar Dados de Ordem de Serviços do MySQL

Este notebook conecta ao banco MySQL, executa um JOIN entre tabelas e exporta os dados para CSV.

## 1. Importar Bibliotecas

In [200]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

## 2. Configurar Credenciais do Banco de Dados

In [201]:
# Configure suas credenciais aqui
config = {
    'host': 'localhost',
    'port': 3306,
    'user': 'root',  # Altere com seu usuário
    'password': '123456',  # Altere com sua senha
    'database': 'grotrack'  # Altere com o nome do seu banco
}

## 3. Conectar ao Banco de Dados

In [202]:
def conectar_mysql(config):
    """Conecta ao banco de dados MySQL"""
    try:
        conexao = mysql.connector.connect(**config)
        
        if conexao.is_connected():
            print("✓ Conexão estabelecida com sucesso!")
            return conexao
    
    except Error as erro:
        print(f"✗ Erro ao conectar ao MySQL: {erro}")
        return None

# Estabelecer conexão
conexao = conectar_mysql(config)

✓ Conexão estabelecida com sucesso!


## 4. Executar Query no Banco de Dados

In [203]:
def buscar_dados(conexao):
    """Executa a query e retorna os dados"""
    try:
        cursor = conexao.cursor(dictionary=True)
        
        query = """
        SELECT os.*, re.data_entrada_prevista, re.data_entrada_efetiva FROM ordem_de_servicos os 
        JOIN registro_entrada re
        ON re.fk_ordem_servico = os.id_ordem_servico
        WHERE os.status = 'FINALIZADO';
        """
        
        cursor.execute(query)
        dados = cursor.fetchall()
        cursor.close()
        
        print(f"✓ {len(dados)} registros recuperados")
        return dados
    
    except Error as erro:
        print(f"✗ Erro ao executar query: {erro}")
        return None

# Executar query
if conexao:
    dados = buscar_dados(conexao)

✓ 37 registros recuperados


## 5. Converter para DataFrame

In [204]:
# Converter para DataFrame
if dados:
    df = pd.DataFrame(dados)
    print(f"\n✓ DataFrame criado com {len(df)} linhas e {len(df.columns)} colunas")
    print(f"\nColunas: {list(df.columns)}")
else:
    print("✗ Nenhum dado para converter")
    df = None


✓ DataFrame criado com 37 linhas e 14 colunas

Colunas: ['id_ordem_servico', 'valor_total', 'valor_total_servicos', 'valor_total_produtos', 'data_saida_prevista', 'data_saida_efetiva', 'data_atualizacao', 'status', 'seguradora', 'nf_realizada', 'pagt_realizado', 'ativo', 'data_entrada_prevista', 'data_entrada_efetiva']


## 6. Visualizar Dados

In [205]:
# Visualizar primeiras linhas
if df is not None:
    display(df.head())

,id_ordem_servico,valor_total,valor_total_servicos,valor_total_produtos,data_saida_prevista,data_saida_efetiva,data_atualizacao,status,seguradora,nf_realizada,pagt_realizado,ativo,data_entrada_prevista,data_entrada_efetiva
0,43,1570.00,1570.00,0.00,2021-01-12,2021-01-12,2026-05-17,FINALIZADO,1,1,1,1,2021-01-09,2021-01-09
1,44,1300.00,1300.00,0.00,2021-01-16,2021-01-16,2026-05-17,FINALIZADO,0,0,1,1,2021-01-14,2021-01-14
2,45,1330.00,1330.00,0.00,2021-01-19,2021-01-19,2026-05-17,FINALIZADO,1,1,0,1,2021-01-17,2021-01-17
3,48,1470.00,1470.00,0.00,2021-03-08,2021-03-07,2026-05-17,FINALIZADO,0,1,1,1,2021-03-03,2021-03-03
4,49,910.00,910.00,0.00,2021-03-11,2021-03-11,2026-05-17,FINALIZADO,1,0,0,1,2021-03-08,2021-03-08


## 7. Informações do DataFrame

In [206]:
# Informações sobre o DataFrame
if df is not None:
    print(df.info())
    print("\nEstatísticas:")
    display(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_ordem_servico       37 non-null     int64 
 1   valor_total            37 non-null     object
 2   valor_total_servicos   37 non-null     object
 3   valor_total_produtos   37 non-null     object
 4   data_saida_prevista    37 non-null     object
 5   data_saida_efetiva     37 non-null     object
 6   data_atualizacao       37 non-null     object
 7   status                 37 non-null     str   
 8   seguradora             37 non-null     int64 
 9   nf_realizada           37 non-null     int64 
 10  pagt_realizado         37 non-null     int64 
 11  ativo                  37 non-null     int64 
 12  data_entrada_prevista  37 non-null     object
 13  data_entrada_efetiva   37 non-null     object
dtypes: int64(5), object(8), str(1)
memory usage: 4.2+ KB
None

Estatísticas:


,id_ordem_servico,seguradora,nf_realizada,pagt_realizado,ativo
count,37.000000,37.000000,37.000000,37.000000,37.0
mean,71.459459,0.594595,0.810811,0.891892,1.0
std,17.100349,0.497743,0.397061,0.314800,0.0
min,43.000000,0.000000,0.000000,0.000000,1.0
25%,57.000000,0.000000,1.000000,1.000000,1.0
50%,72.000000,1.000000,1.000000,1.000000,1.0
75%,88.000000,1.000000,1.000000,1.000000,1.0
max,97.000000,1.000000,1.000000,1.000000,1.0


## 8. Exportar para CSV

In [ ]:
def gerar_csv(df, nome_arquivo='refined/grafana/os_data.csv'):
    """Exporta DataFrame para arquivo CSV"""
    try:
        df.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"✓ Arquivo '{nome_arquivo}' gerado com sucesso!")
        return True
    except Exception as erro:
        print(f"✗ Erro ao gerar CSV: {erro}")
        return False

# Gerar CSV
if df is not None:
    gerar_csv(df, 'refined/grafana/os_data.csv')

✓ Arquivo 'refined/grafana/os_data.csv' gerado com sucesso!


## 9. Fechar Conexão

In [208]:
# Fechar conexão
if conexao and conexao.is_connected():
    conexao.close()
    print("✓ Conexão fechada.")

✓ Conexão fechada.
